# Structured Product Factory — Autocallable Phoenix

**Objective:** Design, price, and risk-manage a worst-of Autocallable Phoenix note on a basket of equities using Monte Carlo simulation.

This notebook demonstrates:
1. Product specification (term sheet)
2. Correlated GBM path simulation (Cholesky decomposition)
3. Monte Carlo pricing with variance estimation
4. Greeks computation via central finite differences (Delta, Gamma, Vega)
5. Correlation sensitivity analysis — *what happens to the bank's P&L if correlations drop?*

In [ ]:
# Colab setup — run this cell if you are on Google Colab
import os
if "COLAB_RELEASE_TAG" in os.environ:
    !git clone --depth 1 https://github.com/louisgay/quant-apps.git /content/quant-apps
    !pip install -q -r /content/quant-apps/structured_product_factory/requirements.txt
    os.chdir("/content/quant-apps/structured_product_factory")

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Ensure engine package is importable
sys.path.insert(0, str(Path.cwd()))

from engine import MarketData, MonteCarloEngine, AutocallablePhoenix, GreeksCalculator

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (12, 5), "font.size": 11})

print("Engine loaded successfully.")

---
## 1. Product Specification

We structure a **3-year worst-of Autocallable Phoenix** linked to **AAPL, MSFT, GOOGL** with the following term sheet:

| Parameter | Value |
|-----------|-------|
| Underlyings | Apple, Microsoft, Alphabet |
| Notional | USD 1,000,000 |
| Maturity | 3 years |
| Observation | Quarterly (12 dates) |
| Autocall Barrier | 100% of initial fixing |
| Coupon Barrier | 65% of initial fixing |
| Coupon | 7.0% p.a. (1.75% per quarter) |
| Put Barrier | 55% of initial fixing |
| Memory | Yes (missed coupons are caught up) |

In [ ]:
product = AutocallablePhoenix(
    underlyings=["AAPL", "MSFT", "GOOGL"],
    notional=1_000_000,
    maturity_years=3.0,
    observation_frequency="quarterly",
    autocall_barrier=1.00,
    coupon_barrier=0.65,
    coupon_rate=0.07,
    put_barrier=0.55,
    memory_feature=True,
)

print(f"Observation dates (year fractions): {product.observation_dates}")
print(f"Coupon per period: {product.coupon_per_period:.2%}")
print(f"Number of observations: {len(product.observation_dates)}")

---
## 2. Market Data

We define the current market environment: spot prices, implied volatilities, a correlation matrix, the risk-free rate, and dividend yields.

In [ ]:
correlation_matrix = np.array([
    [1.00, 0.62, 0.55],   # AAPL
    [0.62, 1.00, 0.58],   # MSFT
    [0.55, 0.58, 1.00],   # GOOGL
])

market_data = MarketData(
    tickers=["AAPL", "MSFT", "GOOGL"],
    spots=np.array([185.0, 420.0, 175.0]),
    volatilities=np.array([0.28, 0.24, 0.30]),
    correlation_matrix=correlation_matrix,
    risk_free_rate=0.04,
    dividend_yields=np.array([0.005, 0.008, 0.0]),
)

# Display correlation matrix
corr_df = pd.DataFrame(
    market_data.correlation_matrix,
    index=market_data.tickers,
    columns=market_data.tickers,
)
print("Correlation Matrix:")
display(corr_df.style.format("{:.2f}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1))

---
## 3. Monte Carlo Pricing

### 3.1 Path Simulation

We simulate **100,000 correlated paths** for the 3 underlyings using a **Cholesky-decomposed GBM** under risk-neutral dynamics:

$$dS_i = (r - q_i) S_i \, dt + \sigma_i S_i \, dW_i, \quad \text{with } dW_i \cdot dW_j = \rho_{ij} \, dt$$

In [ ]:
engine = MonteCarloEngine(n_paths=100_000, seed=42)
paths = engine.simulate(market_data, product.observation_dates)

print(f"Simulated paths shape: {paths.shape}  (paths × assets × dates)")

# Visualize a sample of paths
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
obs = product.observation_dates
times = np.concatenate([[0], obs])

for i, (ax, ticker) in enumerate(zip(axes, market_data.tickers)):
    sample = paths[:200, i, :]  # first 200 paths
    full = np.column_stack([np.full(200, market_data.spots[i]), sample])
    for p in range(200):
        ax.plot(times, full[p], alpha=0.05, color="steelblue", linewidth=0.5)
    ax.axhline(market_data.spots[i] * product.autocall_barrier, color="green",
               linestyle="--", label=f"Autocall ({product.autocall_barrier:.0%})")
    ax.axhline(market_data.spots[i] * product.coupon_barrier, color="orange",
               linestyle="--", label=f"Coupon ({product.coupon_barrier:.0%})")
    ax.axhline(market_data.spots[i] * product.put_barrier, color="red",
               linestyle="--", label=f"Put ({product.put_barrier:.0%})")
    ax.set_title(ticker, fontweight="bold")
    ax.set_xlabel("Time (years)")
    ax.set_ylabel("Price (USD)")
    ax.legend(fontsize=8)

fig.suptitle("Simulated GBM Paths with Barrier Levels", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 3.2 Pricing the Autocallable Phoenix

In [ ]:
result = product.price(paths, market_data.spots, market_data.risk_free_rate)

print("="*60)
print("       AUTOCALLABLE PHOENIX — PRICING RESULTS")
print("="*60)
print(f"  Fair Value:            {result['price']:>14,.0f} USD")
print(f"  Price (% of notional): {result['price_pct']:>14.2f} %")
print(f"  MC Std Error:          {result['std_error']:>14,.0f} USD")
print(f"  Autocall Probability:  {result['autocall_prob']:>14.1%}")
print(f"  Avg Life:              {result['avg_life_years']:>14.2f} years")
print(f"  Capital Loss Prob:     {result['capital_loss_prob']:>14.1%}")
print("="*60)

### 3.3 Convergence Analysis

We verify pricing stability by running the engine at increasing path counts.

In [ ]:
path_counts = [5_000, 10_000, 25_000, 50_000, 100_000, 200_000]
prices = []
errors = []

for n_p in path_counts:
    eng = MonteCarloEngine(n_paths=n_p, seed=42)
    p = eng.simulate(market_data, product.observation_dates)
    r = product.price(p, market_data.spots, market_data.risk_free_rate)
    prices.append(r["price_pct"])
    errors.append(r["std_error"] / product.notional * 100)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(path_counts, prices, "o-", color="steelblue", linewidth=2, label="Price (%)")
ax1.fill_between(path_counts,
                 [p - 1.96*e for p, e in zip(prices, errors)],
                 [p + 1.96*e for p, e in zip(prices, errors)],
                 alpha=0.2, color="steelblue", label="95% CI")
ax1.set_xlabel("Number of Monte Carlo Paths")
ax1.set_ylabel("Price (% of Notional)")
ax1.set_title("Monte Carlo Convergence", fontweight="bold")
ax1.legend()
ax1.set_xscale("log")
plt.tight_layout()
plt.show()

---
## 4. Greeks — Risk Sensitivities

We compute the main Greeks by **central finite differences**:

$$\Delta_i = \frac{V(S_i + h) - V(S_i - h)}{2h}, \quad \Gamma_i = \frac{V(S_i+h) - 2V(S_i) + V(S_i-h)}{h^2}, \quad \mathcal{V}_i = \frac{V(\sigma_i + \epsilon) - V(\sigma_i - \epsilon)}{2\epsilon}$$

In [ ]:
calc = GreeksCalculator(engine, product, market_data)

deltas = calc.delta(bump_pct=0.01)
gammas = calc.gamma(bump_pct=0.01)
vegas  = calc.vega(bump_vol=0.01)

greeks_df = pd.DataFrame({
    "Spot": market_data.spots,
    "Vol": market_data.volatilities,
    "Delta (USD/1pt)": deltas,
    "Gamma (USD/1pt²)": gammas,
    "Vega (USD/1vol pt)": vegas,
})
greeks_df.index = market_data.tickers
greeks_df.index.name = "Underlying"

display(greeks_df.style.format({
    "Spot": "{:.1f}",
    "Vol": "{:.1%}",
    "Delta (USD/1pt)": "{:+,.0f}",
    "Gamma (USD/1pt²)": "{:+,.1f}",
    "Vega (USD/1vol pt)": "{:+,.0f}",
}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
tickers = market_data.tickers
colors = ["#2196F3", "#4CAF50", "#FF9800"]

axes[0].bar(tickers, [deltas[t] for t in tickers], color=colors)
axes[0].set_title("Delta (USD per 1 pt spot move)", fontweight="bold")
axes[0].axhline(0, color="black", linewidth=0.5)

axes[1].bar(tickers, [gammas[t] for t in tickers], color=colors)
axes[1].set_title("Gamma (USD per 1 pt² spot move)", fontweight="bold")
axes[1].axhline(0, color="black", linewidth=0.5)

axes[2].bar(tickers, [vegas[t] for t in tickers], color=colors)
axes[2].set_title("Vega (USD per 1 vol pt)", fontweight="bold")
axes[2].axhline(0, color="black", linewidth=0.5)

fig.suptitle("Greeks Profile", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Interpretation:**
- **Positive Delta**: the product value increases when spots go up (more autocall probability, less downside risk).
- **Negative Vega** (typical for worst-of): higher vol increases the dispersion of the worst performer, making barrier breaches more likely.

---
## 5. Sensitivity Analysis — Correlation Risk

**Why correlation matters for the issuer:**  
Banks issuing autocallables are *short correlation*. When equity correlations drop, the worst performer in the basket diverges further — increasing barrier breach risk, reducing autocall probability, and causing mark-to-market losses on the hedging book.

We reprice the product under uniform correlation shocks from **-20%** to **+20%**.

In [ ]:
bumps = [-0.20, -0.15, -0.10, -0.05, 0.0, 0.05, 0.10, 0.15, 0.20]
corr_sensitivity = calc.correlation_sensitivity(bumps)

# Display as a table
cs_df = pd.DataFrame({
    "Correlation Shock": list(corr_sensitivity.keys()),
    "Price (% of Notional)": list(corr_sensitivity.values()),
})
cs_df["P&L vs Base (USD)"] = (
    (cs_df["Price (% of Notional)"] - corr_sensitivity["+0%"]) * product.notional / 100
)
display(cs_df.style.format({
    "Price (% of Notional)": "{:.2f}",
    "P&L vs Base (USD)": "{:+,.0f}",
}).hide(axis="index"))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

labels = list(corr_sensitivity.keys())
values = list(corr_sensitivity.values())
base_val = corr_sensitivity["+0%"]
bar_colors = ["#e74c3c" if v < base_val else "#2ecc71" for v in values]

bars = ax.bar(labels, values, color=bar_colors, edgecolor="white", linewidth=0.5)
ax.axhline(base_val, color="gray", linestyle="--", linewidth=1, label=f"Base: {base_val:.2f}%")

# Annotate bars
for bar, val in zip(bars, values):
    pnl = (val - base_val) * product.notional / 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{pnl:+,.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xlabel("Correlation Shock", fontsize=12)
ax.set_ylabel("Price (% of Notional)", fontsize=12)
ax.set_title("Correlation Sensitivity — Issuer P&L Impact", fontweight="bold", fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 6. Worst-of Performance Distribution at Maturity

Let's visualize the terminal distribution of the worst performer — the key driver of the product payoff.

In [ ]:
perf = paths / market_data.spots[np.newaxis, :, np.newaxis]
worst_terminal = perf.min(axis=1)[:, -1]  # worst-of at maturity

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(worst_terminal, bins=100, density=True, color="steelblue", alpha=0.7, edgecolor="white")
ax.axvline(product.autocall_barrier, color="green", linestyle="--", linewidth=2,
           label=f"Autocall barrier ({product.autocall_barrier:.0%})")
ax.axvline(product.coupon_barrier, color="orange", linestyle="--", linewidth=2,
           label=f"Coupon barrier ({product.coupon_barrier:.0%})")
ax.axvline(product.put_barrier, color="red", linestyle="--", linewidth=2,
           label=f"Put barrier ({product.put_barrier:.0%})")

ax.set_xlabel("Worst-of Performance (% of initial)", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Terminal Distribution of the Worst Performer", fontweight="bold", fontsize=14)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"P(worst-of < put barrier at maturity | not autocalled) = "
      f"{result['capital_loss_prob']:.1%}")

---
## 7. Autocall Timing Distribution

When does the product get called? This drives the **average life** and impacts the issuer's funding.

In [ ]:
# Recompute autocall dates for visualization
obs_dates = product.observation_dates
worst_perf = perf.min(axis=1)  # (n_paths, n_dates)

autocall_date = np.full(engine.n_paths, np.nan)
called = np.zeros(engine.n_paths, dtype=bool)

for t in range(len(obs_dates)):
    ac = ~called & (worst_perf[:, t] >= product.autocall_barrier)
    autocall_date[ac] = obs_dates[t]
    called[ac] = True

called_dates = autocall_date[~np.isnan(autocall_date)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(called_dates, bins=obs_dates, color="#27ae60", edgecolor="white", alpha=0.8, rwidth=0.85)
ax.set_xlabel("Autocall Date (years)", fontsize=12)
ax.set_ylabel("Number of Paths", fontsize=12)
ax.set_title(f"Autocall Timing Distribution (called: {called.mean():.1%} of paths)",
             fontweight="bold", fontsize=14)
ax.set_xticks(obs_dates)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

---
## 8. Summary

| Metric | Value |
|--------|-------|
| **Pricing** | Monte Carlo with Cholesky-correlated GBM (100k paths) |
| **Greeks** | Central finite differences (Delta, Gamma, Vega) |
| **Key risk** | Short correlation — a 10% drop in pairwise correlation causes significant mark-to-market losses |
| **Use case** | Issuance desks (Leonteq, Vontobel), risk management, structured product sales |

### Next Steps
- Add **local volatility** or **stochastic volatility** (Heston) for smile-consistent pricing
- Implement **AAD (Adjoint Algorithmic Differentiation)** for faster Greeks on larger baskets
- Connect to live market data via Bloomberg/Refinitiv API
- Add **CVA/FVA** adjustments for counterparty and funding risk